# LoRA de estilo WMF (FLUX.3 Image Klein) · IMG-32

Entrena un LoRA de estilo con los banners GPT del juego para usarlo en la Deck (`sd-cli --lora-model-dir`).
Guía completa: `docs/producto/plan-imagenes/lora-kaggle.md` del repo.

**Antes de ejecutar** (panel derecho de Kaggle):
1. *Accelerator*: **GPU T4 x2** (se usa una; bitsandbytes va mejor en T4 que en P100).
2. *Internet*: activado (hace falta el teléfono verificado).
3. *Add-ons → Secrets*: `HF_TOKEN` con un token de lectura de Hugging Face, de la cuenta que aceptó la licencia de
   `black-forest-labs/flux-3-image-klein-base`.
4. *Add data*: el dataset con `wmf-banners.zip` (Kaggle lo descomprime solo).

**Primero el arranque:** deja `ARRANQUE = True` y ejecuta todo (20 pasos). Mira la última celda: memoria máxima,
segundos por paso y que la pérdida no sea `nan`. Si va bien, `ARRANQUE = False` y **Save Version → Save & Run All**
(corre sin el navegador abierto, hasta 12 h). El LoRA sale en *Output*: `lora/pytorch_lora_weights.safetensors`.

In [ ]:
ARRANQUE = True          # True: 20 pasos para ver si cabe y cuánto tarda. False: entrenamiento entero.
PASOS = 20 if ARRANQUE else 1000
RANGO = 16               # rank y alpha del LoRA
DISPARADOR = "wmfstyle"  # palabra que activa el estilo en el prompt

## 1. Dependencias

In [ ]:
!pip install -q "diffusers==0.42.1" "transformers>=4.57.0" "peft>=0.11.1" "accelerate>=0.31.0" "datasets>=3.1.0" \
    bitsandbytes sentencepiece ftfy Jinja2 qwen-vl-utils
!wget -q -O train_lora.py https://raw.githubusercontent.com/huggingface/diffusers/v0.42.1/examples/dreambooth/train_dreambooth_lora_flux3_klein.py
# El repo de FLUX.3 Klein no trae tokenizer/: el tokenizador está en text_encoder/.
!sed -i 's/subfolder="tokenizer"/subfolder="text_encoder"/' train_lora.py && grep -c 'subfolder="text_encoder"' train_lora.py
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Diagnóstico de la pérdida nan: dice si el nan viene de los latentes (VAE), de los textos (Qwen3-VL) o del transformer.
s = open("train_lora.py").read()
A = "    # Caption dropout swaps the empty-prompt embedding"
B = "                # these weighting schemes use a uniform timestep sampling"
assert s.count(A) == 1 and s.count(B) == 1, "El script cambió: no encuentro dónde meter el diagnóstico"
def_est = ('    def _est(nombre, ts):\n'
           '        ts = [t for t in ts if t is not None]\n'
           '        malos = sum(int((~torch.isfinite(t.float())).any()) for t in ts)\n'
           '        mx = max(float(t.float().nan_to_num(0, 0, 0).abs().max()) for t in ts)\n'
           '        print(f"DIAG {nombre}: {malos}/{len(ts)} con nan/inf, |max| {mx:.4g}, dtype {ts[0].dtype}", flush=True)\n')
cache = ('    if args.cache_latents:\n        _est("latentes", instance_latents_cache)\n'
         '    if train_dataset.custom_instance_prompts:\n        _est("textos", prompt_embeds_cache)\n')
paso = ('                if global_step < 3:\n'
        '                    _est(f"paso {global_step} entrada", [model_input])\n'
        '                    _est(f"paso {global_step} textos", [prompt_embeds])\n'
        '                    _est(f"paso {global_step} salida del transformer", [model_pred])\n')
s = s.replace(A, def_est + cache + A).replace(B, paso + B)
open("train_lora.py", "w").write(s)
print("diagnóstico añadido")


In [ ]:
# El transformer en fp32: en fp16 su salida es nan desde el primer paso (la T4 no tiene bf16). El VAE y Qwen3-VL
# siguen en fp16, que no desbordan (lo comprobó el DIAG del 2026-11-02).
s = open("train_lora.py").read()
cambios = [
    ('config_kwargs["bnb_4bit_compute_dtype"] = weight_dtype', 'config_kwargs["bnb_4bit_compute_dtype"] = torch.float32'),
    ('        torch_dtype=weight_dtype,\n    )\n    if args.bnb_quantization_config_path is not None:\n        transformer = prepare',
     '        torch_dtype=torch.float32,\n    )\n    if args.bnb_quantization_config_path is not None:\n        transformer = prepare'),
    ('                with accelerator.autocast():\n                    model_pred = transformer(',
     '                with torch.autocast("cuda", enabled=False):\n                    model_pred = transformer('),
    ('encoder_hidden_states=prompt_embeds,', 'encoder_hidden_states=prompt_embeds.float(),'),
    # accelerator.prepare() envuelve forward en un autocast fp16 que se impone al de arriba: se quita.
    ('    global_step = 0\n',
     '    _t = accelerator.unwrap_model(transformer)\n'
     '    if "_original_forward" in _t.__dict__:\n'
     '        _t.forward = _t.__dict__.pop("_original_forward")\n'
     '    global_step = 0\n'),
]
for viejo, nuevo in cambios:
    assert s.count(viejo) == 1, f"El script cambió: no encuentro {viejo[:60]!r}"
    s = s.replace(viejo, nuevo)
open("train_lora.py", "w").write(s)
print("transformer en fp32")


In [ ]:
# Diagnóstico por capas: en el primer paso, la primera capa del transformer cuya salida tiene nan/inf, y si algún
# peso ya viene con nan/inf.
s = open("train_lora.py").read()
A = "    global_step = 0\n"
assert s.count(A) == 1, "El script cambió: no encuentro dónde meter el diagnóstico por capas"
d = '''    _tr = accelerator.unwrap_model(transformer)
    _malos = [n for n, p in _tr.named_parameters() if not torch.isfinite(p.float()).all()]
    print(f"DIAG pesos con nan/inf: {len(_malos)} {_malos[:5]}", flush=True)
    print("DIAG dtype x_embedder:", _tr.x_embedders["image"].weight.dtype, flush=True)
    _visto, _ganchos = [], []
    def _gancho(nombre):
        def f(mod, ent, sal):
            if _visto:
                return
            t = sal[0] if isinstance(sal, (tuple, list)) else sal
            if torch.is_tensor(t) and t.is_floating_point() and not torch.isfinite(t.detach().float()).all():
                e = [x for x in ent if torch.is_tensor(x) and x.is_floating_point()]
                ent_ok = all(torch.isfinite(x.detach().float()).all() for x in e)
                _visto.append(nombre)
                print(f"DIAG primer nan/inf en {nombre} ({type(mod).__name__}), entrada sana: {ent_ok}, "
                      f"dtype entrada {[x.dtype for x in e]}, salida {t.dtype}", flush=True)
        return f
    for _n, _m in _tr.named_modules():
        _ganchos.append(_m.register_forward_hook(_gancho(_n or "transformer")))
'''
s = s.replace(A, d + A)
B = "                if global_step < 3:\n"
assert s.count(B) == 1, "Falta la celda del diagnóstico anterior"
s = s.replace(B, "                if global_step == 0:\n                    for _g in _ganchos: _g.remove()\n" + B)
open("train_lora.py", "w").write(s)
print("diagnóstico por capas añadido")


## 2. Token de Hugging Face

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import whoami
print("Hugging Face:", whoami()["name"])

## 3. Textos de cada imagen

Un modelo de visión (Qwen3-VL-2B) describe el **contenido** de cada banner (sujeto, lugar, composición, luz), sin
palabras de estilo: el estilo lo aprende la palabra disparadora. Corre en un proceso aparte para que su memoria
se libere antes de entrenar. Si ya existe `metadata.jsonl`, no se repite.

In [ ]:
%%writefile textos.py
import glob, json, os, shutil, sys, torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from PIL import Image
DISP, DST = sys.argv[1], "/kaggle/working/datos"
imgs = sorted(glob.glob("/kaggle/input/**/imagenes/*.png", recursive=True))
assert imgs, "No encuentro las imágenes: ¿has añadido el dataset con wmf-banners.zip?"
os.makedirs(DST, exist_ok=True)
meta = os.path.join(DST, "metadata.jsonl")
if os.path.exists(meta):
    print("metadata.jsonl ya existe:", sum(1 for _ in open(meta)), "textos"); sys.exit()
M = "Qwen/Qwen3-VL-2B-Instruct"
model = Qwen3VLForConditionalGeneration.from_pretrained(M, torch_dtype=torch.float16, device_map="cuda:0")
proc = AutoProcessor.from_pretrained(M)
PIDE = ("Describe what this image shows in one paragraph of 40-70 words: the main subject with its shape, colours "
        "and pose, any other figures, the place, the composition and the light and time of day. Describe only the "
        "content. Do not mention the art style, the medium, the rendering or the image quality.")
with open(meta, "w") as out:
    for i, f in enumerate(imgs):
        nombre = os.path.basename(f); shutil.copy(f, os.path.join(DST, nombre))
        msgs = [{"role": "user", "content": [{"type": "image", "image": Image.open(f).convert("RGB")},
                                              {"type": "text", "text": PIDE}]}]
        ent = proc.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True,
                                       return_tensors="pt").to(model.device)
        with torch.no_grad():
            sal = model.generate(**ent, max_new_tokens=140, do_sample=False)
        txt = proc.batch_decode(sal[:, ent["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
        out.write(json.dumps({"file_name": nombre, "text": f"{DISP}, {txt}"}, ensure_ascii=False) + "\n")
        if i < 3 or i % 20 == 0: print(i, nombre, "→", txt[:120])
print("textos:", len(imgs))

In [ ]:
!python textos.py {DISPARADOR}
!head -c 600 /kaggle/working/datos/metadata.jsonl

## 4. Entrenar

El transformer se carga en 4 bits (nf4) y calcula en fp32: en fp16 su salida es nan y la T4 no tiene bf16.
El VAE y el codificador de texto van en fp16. Los textos y los latentes se calculan una
vez y el codificador de texto se libera antes del primer paso. Se guarda un punto cada 250 pasos.

In [ ]:
import json
# context_embedder sin cuantizar: en nf4 su salida es nan con los textos de Qwen3-VL (|max| 1,1e4).
json.dump({"load_in_4bit": True, "bnb_4bit_quant_type": "nf4", "bnb_4bit_use_double_quant": True,
           "llm_int8_skip_modules": ["context_embedder"]},
          open("nf4.json", "w"))
!rm -rf /kaggle/working/lora && CUDA_VISIBLE_DEVICES=0 accelerate launch --num_processes 1 --mixed_precision fp16 train_lora.py \
  --pretrained_model_name_or_path black-forest-labs/flux-3-image-klein-base \
  --dataset_name /kaggle/working/datos --image_column image --caption_column text --instance_prompt {DISPARADOR} \
  --use_aspect_ratio_buckets --aspect_ratio_buckets "512,1280" \
  --output_dir /kaggle/working/lora --mixed_precision fp16 \
  --bnb_quantization_config_path nf4.json --offload \
  --train_batch_size 1 --gradient_accumulation_steps 1 --gradient_checkpointing --cache_latents \
  --optimizer AdamW --use_8bit_adam --learning_rate 1e-4 --lr_scheduler constant --lr_warmup_steps 0 \
  --rank {RANGO} --lora_alpha {RANGO} --max_train_steps {PASOS} --checkpointing_steps 250 --checkpoints_total_limit 2 \
  --skip_final_inference --seed 42 --report_to tensorboard 2>&1 | tee entrenar.log | grep -vE "it/s\]$" | tail -40

## 5. Resultado del arranque (o del entrenamiento)

In [ ]:
import re, glob, os
log = open("entrenar.log").read()
print(*[l for l in log.splitlines() if l.startswith("DIAG")], sep="\n")
print("pérdidas (últimas):", re.findall(r"loss=([0-9.e+-]+|nan)", log)[-5:])
print("s/paso (últimos):", re.findall(r"(\d+\.\d+)s/it", log)[-3:], "| it/s:", re.findall(r"(\d+\.\d+)it/s", log)[-3:])
print("¿sin memoria?:", "OutOfMemory" in log or "CUDA out of memory" in log)
print("LoRA:", glob.glob("/kaggle/working/lora/*.safetensors"),
      [round(os.path.getsize(f)/1e6, 1) for f in glob.glob("/kaggle/working/lora/*.safetensors")], "MB")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv